# Aircraft Electrical System Health Monitor
## Exploratory Data Analysis & Anomaly Detection Pipeline

**Disclaimer**: This notebook is an educational simulation and prototype for condition-based health monitoring of aircraft electrical buses (115V AC 400Hz and 28V DC). It does not interface with real aircraft systems or certified avionics hardware.

### Pipeline Overview:
1. **Synthetic Telemetry Generation**: Physics-informed time-series representing aircraft electrical operating conditions.
2. **Controlled Fault Injection**: Mathematical injection of 10 electrical and sensor fault modes with strict ground-truth tracking.
3. **Data Preprocessing & Feature Extraction**: Rate of change, deviations, rolling statistics without data leakage.
4. **Engineering Threshold Monitoring**: Deterministic limit envelopes and severity classification.
5. **Isolation Forest Model**: Unsupervised multivariate tree ensemble anomaly detection.
6. **Integrated Health Assessment**: Multi-tiered fusion and continuous health index calculation.
7. **Quantitative Evaluation**: Sample-level, fault-level, and event-level performance metrics.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config
from src.simulator import ElectricalBusSimulator
from src.fault_injection import FaultInjector
from src.preprocessing import DataPreprocessor
from src.threshold_monitor import EngineeringThresholdMonitor
from src.anomaly_detector import IsolationForestDetector
from src.health_monitor import IntegratedHealthMonitor
from src.evaluation import ModelEvaluator

# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["figure.dpi"] = 120

config = load_config(project_root / "config" / "config.yaml")
print(f"Loaded configuration: Bus Type = {config.bus_type.upper()}, Fs = {config.sampling_rate_hz} Hz")

### 1. Baseline Normal Data Generation & Training
We generate a pure, normal baseline dataset representing uncompromised aircraft electrical operations across different load levels (low load, normal load, cruise, transient loading).

In [ ]:
sim = ElectricalBusSimulator(config=config)
preprocessor = DataPreprocessor(config=config)

# Generate 300s baseline training data
df_train = sim.generate(duration_seconds=300.0, bus_type="ac", seed=42)
df_train_proc, X_train_scaled = preprocessor.process(df_train, is_training=True)

print(f"Baseline training observations: {len(X_train_scaled)}")
print("Engineered features used for Isolation Forest:", preprocessor.feature_names)
X_train_scaled.head()

### 2. Fit Unsupervised Isolation Forest Model
The Isolation Forest model is fitted strictly on the normal operating envelope without exposure to ground-truth fault labels.

In [ ]:
iso_detector = IsolationForestDetector(config=config)
iso_detector.fit(X_train_scaled)
print("Isolation Forest fitted successfully on baseline topology.")

### 3. Generate Operational Test Dataset & Inject Faults
We generate a 600-second operational dataset and inject controlled electrical faults (overvoltage, undervoltage, overcurrent, frequency deviations, spikes, sags, sensor dropout, drift, and multivariate anomalies).

In [ ]:
injector = FaultInjector(config=config)
df_test_clean = sim.generate(duration_seconds=600.0, bus_type="ac", seed=100)
df_test, events = injector.inject_faults(df_test_clean, num_faults=10, seed=42)

print(f"Test dataset generated: {len(df_test)} observations.")
print(f"Injected {len(events)} discrete fault events:")
for ev in events:
    print(f"  [{ev.fault_id}] {ev.fault_type:<20} | Time: {ev.start_time:>5.1f}s - {ev.end_time:>5.1f}s | Severity: {ev.severity}")

### 4. Feature Inspection & Correlation Analysis
Let us examine feature distributions and correlations during normal versus fault states.

In [ ]:
df_test_proc, X_test_scaled = preprocessor.process(df_test, is_training=False)

# Correlation matrix across electrical features
corr = X_test_scaled.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Engineered Electrical Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### 5. Multi-Tier Monitoring: Thresholds, ML, and Integrated Health
Run the threshold monitor, Isolation Forest inference, and the integrated health assessment engine.

In [ ]:
thresh_monitor = EngineeringThresholdMonitor(config=config)
df_rules = thresh_monitor.monitor(df_test_proc)

df_ml = iso_detector.predict(X_test_scaled)

health_engine = IntegratedHealthMonitor(config=config)
df_master, alerts = health_engine.evaluate_health(df_test_proc, df_rules, df_ml)

print(f"Total logged alerts: {len(alerts)}")
print("Health Status Breakdown:")
print(df_master["health_status"].value_counts())

### 6. Visualizing Injected Faults vs Detected Anomalies

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

t = df_master["timestamp"]
v = df_master["voltage"]
anom = df_master["final_anomaly"] == 1

ax1.plot(t, v, color="#1E3A8A", alpha=0.7, label="Measured Voltage [V RMS]")
ax1.scatter(t[anom], v[anom], color="#DC2626", s=14, label="Detected Anomaly", zorder=3)
for ev in events:
    ax1.axvspan(ev.start_time, ev.end_time, color="#EF4444", alpha=0.15)
ax1.set_ylabel("Voltage [V RMS]")
ax1.set_title("AC Bus Voltage with Detected Anomalies (Red) & Ground-Truth Intervals (Shaded)")
ax1.legend(loc="upper right")

ax2.plot(t, df_master["health_score"], color="#059669", label="Health Index (0 - 100)")
ax2.axhline(80, color="#F59E0B", linestyle=":", label="Warning Threshold (80)")
ax2.axhline(50, color="#DC2626", linestyle="--", label="Critical Threshold (50)")
for ev in events:
    ax2.axvspan(ev.start_time, ev.end_time, color="#EF4444", alpha=0.15)
ax2.set_xlabel("Time [seconds]")
ax2.set_ylabel("Health Score")
ax2.set_title("Continuous System Health Score Profile")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

### 7. Quantitative Evaluation & Model Benchmark Comparison
Let us honestly evaluate detection efficacy across sample-level, fault-category, and event-level scopes.

In [ ]:
evaluator = ModelEvaluator()

# 1. Benchmark comparison
comp_df = evaluator.compare_methods(df_master)
print("=== Model Benchmark Comparison ===")
display(comp_df[["method_name", "accuracy", "precision", "recall", "f1", "false_positive_rate"]])

# 2. Fault breakdown
fault_df = evaluator.evaluate_by_fault_type(df_master)
print("\n=== Fault Detection Rate Breakdown ===")
display(fault_df)

# 3. Event-level summary
ev_sum = evaluator.evaluate_events(df_master, events)
print(f"\nEvent-Level Detection Rate: {ev_sum.event_detection_rate:.1f}%")
print(f"Average Time-to-First-Detect Latency: {ev_sum.mean_detection_delay_sec:.3f} seconds")